# * Diamond Data Source

In [1]:
import configparser
import datetime as dt
import pandas as pd
import numpy as np
import xlrd
import oracledb
import re

config = configparser.ConfigParser()
config.read('../../my_config.ini')
config.sections()

TDMDBPR_user = config['TDMDBPR']['username']
TDMDBPR_pwd = config['TDMDBPR']['password']
TDMDBPR_db = config['TDMDBPR']['db']
TDMDBPR_host = config['TDMDBPR']['host']
TDMDBPR_port = config['TDMDBPR']['port']

AKPIPRD_user = config['AKPIPRD']['username']
AKPIPRD_pwd = config['AKPIPRD']['password']
AKPIPRD_db = config['AKPIPRD']['db']
AKPIPRD_host = config['AKPIPRD']['host']
AKPIPRD_port = config['AKPIPRD']['port']

curr_dt = dt.datetime.now().date()
str_curr_dt = curr_dt.strftime('%Y%m%d')

In [2]:
''' Input parameter '''

op_dir = '../../data/Adhoc/output'
op_file = f'src_diamond_{str_curr_dt}'

print(f'\nParameter input...\n')
print(f'   -> op_dir: {op_dir}')
print(f'   -> op_file: {op_file}')


Parameter input...

   -> op_dir: ../../data/Adhoc/output
   -> op_file: src_diamond_20260319


## Import Transaction
-> STG_KPI_NEWCO_DIAMOND_ACTUAL_INC

In [3]:
''' Execute transaction '''


# Input parameter
# v_start_date = 20260101
print(f'\nParameter input...')
# print(f'   -> v_start_date: {v_start_date}')

curr_datetime = dt.datetime.now().strftime('%Y-%m-%d, %H:%M:%S')
print(f'\nData as of {curr_datetime}')


# Connect : TDMDBPR
src_dsn = f'{TDMDBPR_user}/{TDMDBPR_pwd}@{TDMDBPR_host}:{TDMDBPR_port}/{TDMDBPR_db}'
src_conn = oracledb.connect(src_dsn)
src_cur = src_conn.cursor()

query = (f"""
    SELECT /*+PARALLEL(8)*/ 
        PAR_KEY, BASENAME, MAX(HD_LOADDATE) AS HD_LOADDATE
        , SUBSTR(TM_KEY_DAY,1,4) AS TM_KEY_YR
        , MIN(TM_KEY_DAY) AS MIN_DAY, MAX(TM_KEY_DAY) AS MAX_DAY
        , CASE 	WHEN REGEXP_LIKE(A.METRIC_NAME, 'Subs Share') THEN 'Subs Share' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Gross Adds Share') THEN 'Gross Add Share' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'New Revenue|Existing Revenue|Other Revenue') THEN 'New & Existing & Other' 
                WHEN REGEXP_LIKE(LOWER(A.METRIC_NAME), 'usage sub|active sub|subbase') THEN 'Subs' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Revenue') THEN 'Revenue' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'AP 1D') THEN 'AP 1D' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Inflow') THEN 'Inflow' 
                WHEN REGEXP_LIKE(LOWER(A.METRIC_NAME), 'gross add|new sub|activation sub') THEN 'Gross Add' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, 'Churn') THEN 'Churn' 
                WHEN REGEXP_LIKE(A.METRIC_NAME, '60DPD') THEN '60DPD' 
                WHEN REGEXP_LIKE(LOWER(A.METRIC_NAME), 'net add') THEN 'Net Add' 
                ELSE '' 
                END METRIC_GRP
        , A.METRIC_CD, A.METRIC_NAME
        , B.METRIC_NAME AS VIS_NAME
        , COMP_CD, VERSION
        , SUM(CASE WHEN AREA_TYPE = 'P' THEN METRIC_VALUE END) AS P_ACTUAL
        , SUM(CASE WHEN AREA_TYPE = 'G' THEN METRIC_VALUE END) AS G
        , SUM(CASE WHEN AREA_TYPE = 'H' THEN METRIC_VALUE END) AS H
        , SUM(CASE WHEN AREA_TYPE = 'HH' THEN METRIC_VALUE END) AS HH
        , SUM(CASE WHEN AREA_TYPE = 'CCAA' THEN METRIC_VALUE END) AS CCAA
        , SUM(CASE WHEN AREA_TYPE = 'CCAATT' THEN METRIC_VALUE END) AS CCAATT
        , MAX(LOAD_DATE) AS LOAD_DATE
    FROM GEOSPCAPPO.STG_KPI_NEWCO_DIAMOND_ACTUAL_INC A
    LEFT JOIN (
        SELECT DISTINCT METRIC_CD, METRIC_NAME, METRIC_GRP
        FROM GEOSPCAPPO.AGG_PERF_NEWCO
        WHERE AREA_CD = 'P'
        AND TM_KEY_DAY >= 20260101
    ) B
        ON B.METRIC_CD = A.METRIC_CD
    GROUP BY PAR_KEY, BASENAME, SUBSTR(TM_KEY_DAY,1,4), A.METRIC_CD, A.METRIC_NAME, B.METRIC_NAME, COMP_CD, VERSION
    --ORDER BY PAR_KEY, BASENAME, 3, A.METRIC_CD
""")


try:
    # Create Dataframe
    src_cur.execute(query)
    rows = src_cur.fetchall()
    src_df = pd.DataFrame.from_records(rows, columns=[x[0] for x in src_cur.description])
    print(f'\nDataFrame: {src_df.shape[0]} rows, {src_df.shape[1]} columns')

    # # Generate CSV file
    # src_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
    # print(f'\n   -> Generate "{op_file}.csv" successfully')

    src_cur.close()


except oracledb.DatabaseError as e:
    print(f'\nError with Oracle : {e}')


finally:
    src_conn.close()


Parameter input...

Data as of 2026-03-19, 10:08:07

DataFrame: 159 rows, 19 columns


## Scan

In [4]:
src_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 19 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   PAR_KEY      159 non-null    int64         
 1   BASENAME     157 non-null    object        
 2   HD_LOADDATE  159 non-null    datetime64[ns]
 3   TM_KEY_YR    159 non-null    object        
 4   MIN_DAY      159 non-null    int64         
 5   MAX_DAY      159 non-null    int64         
 6   METRIC_GRP   147 non-null    object        
 7   METRIC_CD    159 non-null    object        
 8   METRIC_NAME  159 non-null    object        
 9   VIS_NAME     85 non-null     object        
 10  COMP_CD      159 non-null    object        
 11  VERSION      159 non-null    object        
 12  P_ACTUAL     159 non-null    float64       
 13  G            156 non-null    float64       
 14  H            159 non-null    float64       
 15  HH           159 non-null    float64       
 16  CCAA    

In [5]:
missing_count = src_df.isnull().sum()
missing_count

PAR_KEY          0
BASENAME         2
HD_LOADDATE      0
TM_KEY_YR        0
MIN_DAY          0
MAX_DAY          0
METRIC_GRP      12
METRIC_CD        0
METRIC_NAME      0
VIS_NAME        74
COMP_CD          0
VERSION          0
P_ACTUAL         0
G                3
H                0
HH               0
CCAA           159
CCAATT         159
LOAD_DATE        0
dtype: int64

In [6]:
# src_df[['METRIC_GRP', 'METRIC_CD', 'METRIC_NAME']].drop_duplicates()

src_df['METRIC_GRP'].drop_duplicates()

0            Gross Add Share
1                      AP 1D
2                  Gross Add
3                     Inflow
4                       Subs
7                 Subs Share
10                      None
16    New & Existing & Other
20                     60DPD
22                     Churn
37                   Net Add
47                   Revenue
Name: METRIC_GRP, dtype: object

## Obsolete Data

In [7]:
obsolete_df = src_df.query(" METRIC_GRP in ('Inflow','Gross Add','AP 1D','New & Existing & Other') ")
obsolete_df

,PAR_KEY,BASENAME,HD_LOADDATE,TM_KEY_YR,MIN_DAY,MAX_DAY,METRIC_GRP,METRIC_CD,METRIC_NAME,VIS_NAME,COMP_CD,VERSION,P_ACTUAL,G,H,HH,CCAA,CCAATT,LOAD_DATE
1,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,AP 1D,DB1S000109AE,Prepaid VAS Day1 / AP 1D : DTAC : Direct Sales,None,DTAC,A,1.005026e+06,1.005026e+06,1.005026e+06,1.005026e+06,None,None,2026-03-17
2,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,Gross Add,DB1S000101AG,Prepaid Gross Add : DTAC : Modern Trade,None,DTAC,A,2.631700e+04,2.808200e+04,2.672500e+04,2.672500e+04,None,None,2026-03-17
3,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,Inflow,DB0R00010001,Total Inflow M1 : DTAC,None,DTAC,A,4.094282e+07,4.134696e+07,8.190564e+07,4.096915e+07,None,None,2026-03-17
6,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,Inflow,DB1R001000AG,Prepaid Inflow M2 : DTAC : Modern Trade,None,DTAC,A,2.515215e+06,2.511906e+06,2.514669e+06,2.514669e+06,None,None,2026-03-17
9,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,AP 1D,DB1S000109,Prepaid VAS Day1 / AP 1D : DTAC,None,DTAC,A,4.143750e+07,4.143036e+07,4.144278e+07,4.144278e+07,None,None,2026-03-17
11,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,AP 1D,DB1S000109AC,Prepaid VAS Day1 / AP 1D : DTAC : Branded Retail,None,DTAC,A,5.974615e+06,5.974615e+06,5.974615e+06,5.974615e+06,None,None,2026-03-17
16,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,New & Existing & Other,DB1R000101,Prepay New Revenue,Prepaid Revenue : DTAC (D-1),DTAC,A,7.764394e+07,7.764394e+07,7.761377e+07,7.761377e+07,None,None,2026-03-17
17,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,Gross Add,DB1S000101AK,Prepaid Gross Add : DTAC : Wholesales,None,DTAC,A,5.276400e+04,7.342100e+04,5.323200e+04,5.323200e+04,None,None,2026-03-17
18,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,Gross Add,DB2S020101,Postpaid Gross Add (Exclude M2M) B2B : DTAC,None,DTAC,A,1.156000e+03,1.202000e+03,1.156000e+03,1.156000e+03,None,None,2026-03-17
21,20260317,D_CORPKPI_ACTUAL_01_20260317.txt.gpg,2026-03-18 19:54:11.145,2026,20260308,20260317,Inflow,DB1R000900AJ,Prepaid Inflow M1 : DTAC : Retail Sales,None,DTAC,A,2.030737e+07,2.030737e+07,2.032342e+07,2.032342e+07,None,None,2026-03-17


## Output

In [8]:
''' Generate CSV file '''

src_df.to_csv(f'{op_dir}/{op_file}.csv', index=False, encoding='utf-8')
print(f'\n   -> Generate "{op_file}.csv" successfully')


   -> Generate "src_diamond_20260319.csv" successfully
